In [2]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [3]:
class TrackedObject:
    def __init__(self, **kwargs):
        # history добавляем напрямую, чтобы не логировать
        super().__setattr__('history', [])

        # записываем входные параметры как атрибуты
        for k, v in kwargs.items():
            self.__setattr__(k, v)

    def __setattr__(self, name, value):
        before_size = len(self.__dict__)
        self.__dict__[name] = value
        after_size = len(self.__dict__)
        self.history.append(
            f"SET {name} = {value}; dict size: {before_size} -> {after_size}"
        )

    def __delattr__(self, name):
        before_size = len(self.__dict__)
        if name in self.__dict__:
            del self.__dict__[name]
        after_size = len(self.__dict__)
        self.history.append(
            f"DEL {name}; dict size: {before_size} -> {after_size}"
        )

obj = TrackedObject(x=1, y=2)
obj.z = 3                # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x

print(obj.__dict__)
for event in obj.history:
  print(event)

{'history': ['SET x = 1; dict size: 1 -> 2', 'SET y = 2; dict size: 2 -> 3', 'SET z = 3; dict size: 3 -> 4', 'SET __str__ = <function <lambda> at 0x1123c9170>; dict size: 4 -> 5', 'DEL x; dict size: 5 -> 4'], 'y': 2, 'z': 3, '__str__': <function <lambda> at 0x1123c9170>}
SET x = 1; dict size: 1 -> 2
SET y = 2; dict size: 2 -> 3
SET z = 3; dict size: 3 -> 4
SET __str__ = <function <lambda> at 0x1123c9170>; dict size: 4 -> 5
DEL x; dict size: 5 -> 4


## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [5]:
def merge(seqs: list[list]) -> list:
    result = []
    seqs = [list(s) for s in seqs if s]
    while seqs:
        for seq in seqs:
            candidate = seq[0]
            # проверяем, что кандидат не встречается в хвостах других списков
            if not any(candidate in s[1:] for s in seqs):
                break
        else:
            raise TypeError("Невозможно построить MRO из-за циклической зависимости")

        result.append(candidate)

        new_seqs = []
        for seq in seqs:
            if seq[0] == candidate:
                seq = seq[1:]
            else:
                seq = [x for x in seq if x != candidate]
            if seq:
                new_seqs.append(seq)

        seqs = new_seqs

    return result

def c3_linearize(cls: type) -> list[type]:
    # если нет родителей, то MRO это просто класс
    if not cls.__bases__:
        return [cls]

    bases_mro = [c3_linearize(base) for base in cls.__bases__]
    bases = [list(mro) for mro in bases_mro]
    bases.append(list(cls.__bases__))
    return [cls] + merge(bases)

class A:
    def action(self):
        print("A")


class B(A):
    def action(self):
        print("B")
        super().action()


class C(A):
    def action(self):
        print("C")
        super().action()


class D(B, C):
    def action(self):
        print("D")
        super().action()


class E(A):
    def action(self):
        print("E")
        super().action()


class F(B, E):
    def action(self):
        print("F")
        super().action()


class G(D, F):
    def action(self):
        print("G")
        super().action()
    

def show(cls):
    custom = c3_linearize(cls)
    builtin = list(cls.__mro__)

    print(f"\nClass {cls.__name__}")
    print("Custom C3:", [c.__name__ for c in custom])
    print("CPython MRO:", [c.__name__ for c in builtin])


show(G)
show(F)
show(D)


Class G
Custom C3: ['G', 'D', 'F', 'B', 'C', 'E', 'A', 'object']
CPython MRO: ['G', 'D', 'F', 'B', 'C', 'E', 'A', 'object']

Class F
Custom C3: ['F', 'B', 'E', 'A', 'object']
CPython MRO: ['F', 'B', 'E', 'A', 'object']

Class D
Custom C3: ['D', 'B', 'C', 'A', 'object']
CPython MRO: ['D', 'B', 'C', 'A', 'object']


## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [6]:
class SecureBase:
    def __init__(self):
        self.__secret_value = "BASE_SECRET"
        self._semi_private = "BASE_SEMI"
        self.public = "BASE_PUBLIC"


class SecureChild(SecureBase):
    def __init__(self):
        super().__init__()

        self.__secret_value = "CHILD_SECRET"
        self._semi_private = "CHILD_SEMI"
        self.public = "CHILD_PUBLIC"

    def dump_dict(self):
        return self.__dict__


base = SecureBase()
child = SecureChild()

print("BASE __dict__:")
print(base.__dict__)

print("\nCHILD __dict__:")
print(child.__dict__)

print("Access mangled from outside:")
print(base._SecureBase__secret_value)
print(child._SecureChild__secret_value)
print(child._SecureBase__secret_value)

BASE __dict__:
{'_SecureBase__secret_value': 'BASE_SECRET', '_semi_private': 'BASE_SEMI', 'public': 'BASE_PUBLIC'}

CHILD __dict__:
{'_SecureBase__secret_value': 'BASE_SECRET', '_semi_private': 'CHILD_SEMI', 'public': 'CHILD_PUBLIC', '_SecureChild__secret_value': 'CHILD_SECRET'}
Access mangled from outside:
BASE_SECRET
CHILD_SECRET
BASE_SECRET


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [9]:
class NoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")

    def __init__(self, x):
        self.x = x

def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = None
        d["dict_keys"] = None
    return d

n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

# Попытка динамического добавления атрибутов
n.z = 10
print("NoSlots z:", n.z)

try:
    w.z = 10
except Exception as e:
    print("WithSlots error:", e)

try:
    ww.z = 10
except Exception as e:
    print("WithSlotsWeak error:", e)

# weakref

r1 = weakref.ref(n)
print("NoSlots weakref:", r1())

r2 = None
try:
    r2 = weakref.ref(w)
except Exception as e:
    print("WithSlots weakref error:", e)

print("NoSlots weakref:", r1())

if r2 is not None:
    print("WithSlots weakref:", r2())
else:
    print("WithSlots weakref: not supported")

NoSlots z: 10
WithSlots error: 'WithSlots' object has no attribute 'z' and no __dict__ for setting new attributes
WithSlotsWeak error: 'WithSlotsWeak' object has no attribute 'z' and no __dict__ for setting new attributes
NoSlots weakref: <__main__.NoSlots object at 0x11240c1a0>
WithSlots weakref error: cannot create weak reference to 'WithSlots' object
NoSlots weakref: <__main__.NoSlots object at 0x11240c1a0>
WithSlots weakref: not supported


## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [11]:
class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name

        if parent is not None:
            if weak_parent:
                self.parent = weakref.ref(parent)
            else:
                self.parent = parent

    def get_parent(self):
        if isinstance(self.parent, weakref.ref):
            return self.parent()
        return self.parent


def refcount(obj):
    return sys.getrefcount(obj) - 1


# Цикл со сильными ссылками
a = Node("A")
b = Node("B")

a.parent = b
b.parent = a

print("Strong cycle refcounts:", refcount(a), refcount(b))

del a, b
gc.collect()   # объекты должны быть собраны как циклический мусор

# Цикл с weakref
c = Node("C")
d = Node("D")

c.parent = weakref.ref(d)
d.parent = weakref.ref(c)

print("Weak cycle refcounts:", refcount(c), refcount(d))

wr_c = weakref.ref(c)
print("wr_c before:", wr_c())

del c
gc.collect()

print("wr_c after deletion:", wr_c())

Strong cycle refcounts: 2 2
Weak cycle refcounts: 1 1
wr_c before: <__main__.Node object at 0x110b54cd0>
wr_c after deletion: None
